# A1 — Ground truth: folding and binding free energies (Stage 1)

**Question.** What are the measured effects to learn from, and do they mean what we think they
mean (the sign, the scale)?

**What it does.**
- **ddPCA** (Faure et al. 2022): MoCHI separates each variant's ΔΔG of folding from its ΔΔG of
  binding for GRB2-SH3, PSD95-PDZ3 and GB1. Bootstrap spread flags unreliable fits.
- **MegaScale** (Tsuboyama et al. 2023): 389,068 single-substitution folding stabilities.
  Shipped with the opposite sign, so negated. SH3 domains identified by ≥50 % identity.
- **Sign and scale check** against independent in vitro measurements, before anything is
  trained. The binding check needs a numbering offset, which is derived, not assumed.
- **The premise, in measured data:** buried residues cost folding energy, interface residues
  cost binding energy.

**Inputs:** `input/datasets/Data.zip`, `Processed_K50_dG_datasets.zip`, `input/uniprot/sh3_domains_human.tsv`.
**Outputs:** `ddpca_ground_truth.parquet`, `megascale_singles.parquet`, `megascale_sh3_matches.csv`,
`calibration.csv`, `separation_table.csv`, `baseline_*.csv`, `RESULTS_A1.md`.
**Runtime:** about 10 min, CPU. Explanation: `docs/A1_ground_truth.md`.

Convention everywhere: **positive = destabilising / weaker binding**.

In [1]:
# ── setup (identical in every notebook) ─────────────────────────────────────────
ROOT   = '/content/drive/MyDrive/sh3_repro'   # the Drive folder from UPLOAD_GUIDE.md
RESUME = True                                  # reuse finished steps after a disconnect

import os, sys, json, time, importlib
from google.colab import drive; drive.mount('/content/drive')
os.system('pip -q install biopython')
sys.path.insert(0, f'{ROOT}/code')
import sh3_common as C; importlib.reload(C)
import numpy as np, pandas as pd
R = C.Run(ROOT, 'A1', 'Ground truth', resume=RESUME)
print(C.env_info())

Mounted at /content/drive
A1 — Ground truth
  results -> /content/drive/MyDrive/sh3_repro/results/A1_ground_truth
{'python': '3.13.15', 'numpy': '2.1.3', 'pandas': '2.2.3', 'scipy': '1.16.3', 'sklearn': '1.6.1', 'transformers': '5.16.1', 'Bio': '1.88', 'torch': '2.11.0+cpu', 'gpu': 'none'}


In [2]:
import glob, collections
from Bio import Align
R.verify_inputs(['input/datasets/Data.zip', 'input/datasets/Processed_K50_dG_datasets.zip',
                 'input/uniprot/sh3_domains_human.tsv'])
DOMAINS = ['GRB2-SH3', 'PSD95-PDZ3', 'GB1']
FIT = 'mochi__fit_tmodel_3state_sparse_dimsum128'
m = R.extract('input/datasets/Data.zip', lambda s: any(s.startswith(f'Data/mochi/{d}/{FIT}/') for d in DOMAINS)
              or s.startswith(('Data/pdb/', 'Data/in_vitro/', 'Data/foldx/', 'Data/EVE/', 'Data/MSA/')))
print(f'Faure archive: {len(m)} files')
m = R.extract('input/datasets/Processed_K50_dG_datasets.zip',
              lambda s: s.endswith('/Tsuboyama2023_Dataset2_Dataset3_20230416.csv'))
assert len(m) == 1, m                     # guards against the macOS '._' shadow copy
TSU = f'{R.WORK}/{m[0]}'; W = R.WORK

  verified input/datasets/Data.zip  (13s)
  verified input/datasets/Processed_K50_dG_datasets.zip  (13s)
  verified input/uniprot/sh3_domains_human.tsv  (0s)
Faure archive: 492 files


## 1. ddPCA: separate folding and binding energies, with reliability flags

In [3]:
PAT = C.MUT_PATTERN
def load_domain(dom):
    base = f'{W}/Data/mochi/{dom}/{FIT}'
    w = pd.read_csv(f'{base}/model_weights_0.txt', sep='\t')
    p = w.id.str.extract(PAT); p.columns = ['wt_aa', 'pos', 'mut_aa']
    d = pd.concat([w, p], axis=1).dropna(subset=['wt_aa']).copy(); d['pos'] = d.pos.astype(int); d['domain'] = dom
    boots = sorted(glob.glob(f'{base}/bootstrap/boot_*/model_weights_0.txt'))
    F = pd.concat([pd.read_csv(b, sep='\t').set_index('id')['folding_coefficient'] for b in boots], axis=1)
    B = pd.concat([pd.read_csv(b, sep='\t').set_index('id')['binding_coefficient'] for b in boots], axis=1)
    return (d.set_index('id').join(F.std(axis=1).rename('folding_sd')).join(B.std(axis=1).rename('binding_sd'))
            .reset_index().assign(n_bootstraps=len(boots)))
dd = pd.concat([load_domain(d) for d in DOMAINS], ignore_index=True)
# a binding coefficient means nothing for a variant too unstable to fold: those fits are unreliable
dd['binding_reliable'] = dd.binding_coefficient.notna() & (dd.binding_coefficient.abs() <= 4) & (dd.binding_sd <= 0.5)
dd['folding_reliable'] = dd.folding_coefficient.notna() & (dd.folding_coefficient.abs() <= 6) & (dd.folding_sd <= 0.5)
QD = dd.groupby('domain').agg(singles=('id', 'size'), with_binding=('binding_coefficient', lambda s: s.notna().sum()),
                              reliable_binding=('binding_reliable', 'sum'), reliable_folding=('folding_reliable', 'sum'),
                              positions=('pos', 'nunique'), bootstraps=('n_bootstraps', 'first'),
                              duplicated_ids=('id', lambda s: int(s.duplicated().sum()))).reset_index()
print(QD.to_string(index=False))
assert QD.duplicated_ids.sum() == 0

    domain  singles  with_binding  reliable_binding  reliable_folding  positions  bootstraps  duplicated_ids
       GB1     1045          1045              1000               878         55          10               0
  GRB2-SH3     1056           756               619              1032         56          10               0
PSD95-PDZ3     1587          1567              1221              1199         84          10               0


## 2. Sign and scale check against in vitro measurements
Folding against Φ-value kinetics of GRB2-SH3 (Troilo 2018); binding against double-mutant-cycle ΔΔG of PSD95-PDZ3 (Chi 2008). Chi numbers residues on full-length DLG4, so the offset to the MoCHI construct is found by maximising wild-type agreement.

In [4]:
lit = pd.read_csv(f'{W}/Data/in_vitro/GRB2_literature_free_energies.txt', sep='\t')
tro = lit[lit.Dataset == 'Troilo_2018'].copy(); tro['id'] = tro.id.str.strip()
K = pd.Series((tro.Kf / tro.Ku).values, index=tro.id)
lf = (-0.592 * np.log(K / K['WT'])).drop('WT')
fold_pairs = pd.concat([lf.rename('in_vitro'), dd[dd.domain == 'GRB2-SH3'].set_index('id').folding_coefficient], axis=1).dropna()
r_fold = fold_pairs.corr().iloc[0, 1]
chi = pd.read_csv(f'{W}/Data/in_vitro/PDZ_literature_free_energies/Chi2008_invitro_kd_ddGf_ddGb_PDZ_CRIPTpeptide.txt', sep=r'\s+')
cm = chi.id.str.strip().str.extract(PAT).dropna(); cm[1] = cm[1].astype(int)
pz = dd[dd.domain == 'PSD95-PDZ3']; wt_at = dict(zip(pz.pos, pz.wt_aa))
off = max(range(250, 360), key=lambda o: sum(wt_at.get(p - o) == w for w, p in zip(cm[0], cm[1])))
chi.loc[cm.index, 'mid'] = [f'{w}{p - off}{m}' for w, p, m in zip(cm[0], cm[1], cm[2])]
bind_pairs = chi.dropna(subset=['mid']).set_index('mid')[['b_ddg']].join(pz.set_index('id').binding_coefficient).dropna()
r_bind = bind_pairs.corr().iloc[0, 1]
print(f'folding r {r_fold:+.3f} (n={len(fold_pairs)}) | binding r {r_bind:+.3f} (n={len(bind_pairs)}) | Chi offset {off}')
assert r_fold > 0 and r_bind > 0, 'MoCHI sign disagrees with in vitro data - stop'
pd.DataFrame([dict(axis='folding', reference='Troilo 2018 (GRB2-SH3, Kf/Ku)', n=len(fold_pairs), pearson_r=r_fold),
              dict(axis='binding', reference='Chi 2008 (PSD95-PDZ3, b_ddg)', n=len(bind_pairs), pearson_r=r_bind,
                   numbering_offset=off)]).to_csv(R.out('calibration.csv'), index=False)

folding r +0.829 (n=15) | binding r +0.612 (n=12) | Chi offset 310


## 3. MegaScale folding stabilities
Single substitutions only; the sign is flipped on load. A MegaScale domain is an SH3 if it is ≥50 % identical to a human SH3 domain.

In [5]:
parts = []
for ch in pd.read_csv(TSU, chunksize=200_000, low_memory=False,
                      usecols=['WT_name', 'aa_seq', 'mut_type', 'ddG_ML', 'dG_ML', 'deltaG', 'deltaG_95CI', 'WT_cluster']):
    ch = ch[ch.mut_type.astype(str).str.match(r'^[A-Z]\d+[A-Z]$')]
    parts.append(ch[pd.to_numeric(ch.ddG_ML, errors='coerce').notna()])
mega = pd.concat(parts, ignore_index=True)
mega['ddG_fold'] = -pd.to_numeric(mega.ddG_ML)          # shipped with the opposite sign
DOMTBL = pd.read_csv(f'{R.IN}/uniprot/sh3_domains_human.tsv', sep='\t')
idal = Align.PairwiseAligner(open_gap_score=-11, extend_gap_score=-1, substitution_matrix=None)
idal.match_score = 1; idal.mismatch_score = 0
def identity(a, b):
    if abs(len(a) - len(b)) > 25: return 0.0
    try: return idal.score(a, b) / min(len(a), len(b))
    except Exception: return 0.0
hits = []
for _, m_ in mega.drop_duplicates('WT_name')[['WT_name', 'aa_seq']].iterrows():
    best = (0, None)
    for _, d in DOMTBL[['gene', 'uniprot', 'domain_idx', 'domain_seq']].iterrows():
        s = identity(str(m_.aa_seq), d.domain_seq)
        if s > best[0]: best = (s, d)
    if best[0] >= 0.50:
        hits.append(dict(WT_name=m_.WT_name, gene=best[1].gene, uniprot=best[1].uniprot,
                         domain_idx=best[1].domain_idx, identity=round(best[0], 3)))
hits = pd.DataFrame(hits)
mega['is_sh3'] = mega.WT_name.isin(set(hits.WT_name))
mega = mega.merge(hits.drop(columns='identity'), on='WT_name', how='left')
print(f'MegaScale: {len(mega):,} single substitutions, {mega.WT_name.nunique()} domains, {len(hits)} SH3 structures '
      f'({hits.gene.nunique()} genes, {int(mega.is_sh3.sum()):,} variants)')
print(f'median ddG_fold after the sign flip: {mega.ddG_fold.median():+.2f} (positive = destabilising)')
mega[['WT_name', 'aa_seq', 'mut_type', 'ddG_fold', 'is_sh3', 'gene', 'uniprot', 'domain_idx']].to_parquet(
    R.out('megascale_singles.parquet'), index=False)
hits.sort_values('identity', ascending=False).to_csv(R.out('megascale_sh3_matches.csv'), index=False)
dd.to_parquet(R.out('ddpca_ground_truth.parquet'), index=False)

MegaScale: 389,068 single substitutions, 412 domains, 26 SH3 structures (15 genes, 27,590 variants)
median ddG_fold after the sign flip: +0.49 (positive = destabilising)


## 4. The premise in measured data: burial drives folding cost, the interface drives binding cost
GRB2-SH3 variants by structural class from the solved peptide complex (2VWF): buried = relative solvent accessibility < 0.25, interface = within 5 Å of the peptide. No model is involved.

In [6]:
def parse_struct(pdb, rsasa_pdb, dom_chain='A', pep_chain='B'):
    rs = {int(l[22:26]): float(l[60:66]) for l in open(rsasa_pdb) if l.startswith('ATOM') and l[21] == dom_chain}
    dom, pep = collections.defaultdict(list), []
    for l in open(pdb):
        if not l.startswith('ATOM'): continue
        xyz = (float(l[30:38]), float(l[38:46]), float(l[46:54]))
        if l[21] == dom_chain: dom[int(l[22:26])].append(xyz)
        elif l[21] == pep_chain: pep.append(xyz)
    mind = {r: min(((a[0]-b[0])**2 + (a[1]-b[1])**2 + (a[2]-b[2])**2) ** .5 for a in ats for b in pep) if pep else np.nan
            for r, ats in dom.items()}
    return pd.DataFrame({'pos': list(rs), 'rsasa': [rs[p] for p in rs], 'dist_peptide': [mind.get(p, np.nan) for p in rs]})
st = parse_struct(f'{W}/Data/pdb/2vwf.pdb', f'{W}/Data/pdb/2vwf_RSASA.pdb')
g = dd[dd.domain == 'GRB2-SH3'].merge(st, on='pos', how='left')
g['buried'] = g.rsasa < 0.25; g['interface'] = g.dist_peptide < 5.0
SEP = g.groupby(['buried', 'interface'])[['folding_coefficient', 'binding_coefficient']].median()
SEP['variants'] = g.groupby(['buried', 'interface']).size()
SEP = SEP.reset_index(); SEP.to_csv(R.out('separation_table.csv'), index=False); print(SEP.round(3).to_string(index=False))

 buried  interface  folding_coefficient  binding_coefficient  variants
  False      False                0.868               -0.036       524
  False       True                0.569                0.645       228
   True      False                3.566                0.141       247
   True       True                1.279                2.900        57


## 5. Conventional baselines shipped with the ddPCA archive (FoldX, EVE, MSA conservation)

In [7]:
import re
FOLDX = {'GRB2-SH3': 'PS_2vwf_nowater_noligand_restrict_scanning_output.txt',
         'PSD95-PDZ3': 'PS_1be9_nowater_noligand_restrict_scanning_output.txt',
         'GB1': 'PS_2gb1_nowater_noligand_restrict_scanning_output.txt'}
AA3 = {'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I',
       'LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V'}
fx = []
for dom, fn in FOLDX.items():
    for line in open(f'{W}/Data/foldx/{dom}/{fn}'):
        p = line.split()
        if len(p) != 2: continue
        m_ = re.match(r'^([A-Z]{3})([A-Za-z])(\d+)([A-Z])$', p[0])
        if m_: fx.append({'domain': dom, 'id': f'{AA3.get(m_.group(1), "X")}{m_.group(3)}{m_.group(4)}', 'foldx_ddg': float(p[1])})
foldx = pd.DataFrame(fx); foldx.to_csv(R.out('baseline_foldx.csv'), index=False)
chk = dd.merge(foldx, on=['domain', 'id']).dropna(subset=['folding_coefficient', 'foldx_ddg'])
r_foldx = chk.folding_coefficient.corr(chk.foldx_ddg)
EVE = {'GRB2-SH3': 'GRB2_HUMAN_149-217_11-26-2021_b02_single_mutant_matrix.csv',
       'PSD95-PDZ3': 'DLG4_HUMAN_301-404_11-26-2021_b04_single_mutant_matrix.csv',
       'GB1': 'SPG1_STRSG_218-292_11-26-2021_b05_single_mutant_matrix.csv'}
ev = []
for dom, fn in EVE.items():
    e = pd.read_csv(f'{W}/Data/EVE/{dom}/{fn}', on_bad_lines='skip', low_memory=False)
    e = e[[c for c in ['id', 'prediction_EVE_evol_indices', 'prediction_epistatic', 'column_conservation'] if c in e.columns]].copy()
    e['domain'] = dom; ev.append(e)
pd.concat(ev, ignore_index=True).dropna(subset=['id']).to_csv(R.out('baseline_eve.csv'), index=False)
pd.concat([pd.read_csv(f'{W}/Data/MSA/{d}/frequencies.csv').assign(domain=d) for d in DOMAINS],
          ignore_index=True).to_csv(R.out('baseline_msa_conservation.csv'), index=False)
print(f'FoldX vs MoCHI folding: r {r_foldx:+.3f} over {len(chk):,} variants')

FoldX vs MoCHI folding: r +0.494 over 1,024 variants


## 6. Results

In [8]:
for k, v in [('ddpca_singles', len(dd)),
             ('ddpca_both_energies', int((dd.folding_coefficient.notna() & dd.binding_coefficient.notna()).sum())),
             ('reliable_binding', int(dd.binding_reliable.sum())), ('reliable_folding', int(dd.folding_reliable.sum())),
             *[(f'reliable_binding_{d}', int(dd[dd.domain == d].binding_reliable.sum())) for d in DOMAINS],
             ('calibration_fold_r', r_fold), ('calibration_fold_n', len(fold_pairs)),
             ('calibration_bind_r', r_bind), ('calibration_bind_n', len(bind_pairs)), ('chi_offset', off),
             ('megascale_singles', len(mega)), ('megascale_domains', mega.WT_name.nunique()),
             ('megascale_sh3_structures', len(hits)), ('megascale_sh3_genes', hits.gene.nunique()),
             ('megascale_sh3_variants', int(mega.is_sh3.sum())),
             ('foldx_overlap', len(chk)), ('foldx_vs_mochi_r', r_foldx)]:
    R.metric(k, v)
lab = lambda b, i: f"{'buried' if b else 'surface'}_{'interface' if i else 'not_interface'}"
for _, r in SEP.iterrows():
    R.metric(f'sep_{lab(r.buried, r.interface)}_fold', r.folding_coefficient)
    R.metric(f'sep_{lab(r.buried, r.interface)}_bind', r.binding_coefficient)
R.section('ddPCA per domain', 'Reliable binding: coefficient present, |ΔΔG_bind| ≤ 4 and bootstrap SD ≤ 0.5.', QD)
R.section('In vitro calibration', 'Both correlations must be positive; they are the pipeline-wide sign check.',
          pd.read_csv(R.out('calibration.csv')))
R.section('Measured separation (GRB2-SH3, median ΔΔG in kcal/mol)',
          'The empirical premise of the thesis: the two energies separate along structural lines.', SEP, '{:.3f}')
R.section('MegaScale domains matched to human SH3 domains', '', hits.sort_values('identity', ascending=False))
R.finish([f"ddPCA: {len(dd):,} single substitutions in 3 domains; {R.metrics['reliable_binding']:,} reliable binding energies "
          f"(GRB2-SH3 {R.metrics['reliable_binding_GRB2-SH3']}, the only SH3).",
          f"MegaScale: {len(mega):,} folding stabilities in {mega.WT_name.nunique()} domains, sign flipped; "
          f"{len(hits)} structures of {hits.gene.nunique()} human SH3 genes ({R.metrics['megascale_sh3_variants']:,} variants).",
          f"Sign check: folding r {r_fold:+.3f} (n={len(fold_pairs)}), binding r {r_bind:+.3f} (n={len(bind_pairs)}, offset {off}).",
          f"Measured: buried non-interface residues cost {SEP.iloc[2].folding_coefficient:.2f} kcal/mol of folding energy; "
          f"buried interface residues cost {SEP.iloc[3].binding_coefficient:.2f} of binding energy."])


A1: REPRODUCED
                      quantity      this_run   recorded           status
                 ddpca_singles   3688.000000   3688.000        identical
           ddpca_both_energies   3368.000000   3368.000        identical
              reliable_binding   2840.000000   2840.000        identical
              reliable_folding   3109.000000   3109.000        identical
     reliable_binding_GRB2-SH3    619.000000    619.000        identical
   reliable_binding_PSD95-PDZ3   1221.000000   1221.000        identical
          reliable_binding_GB1   1000.000000   1000.000        identical
            calibration_fold_r      0.829057      0.829 within tolerance
            calibration_fold_n     15.000000     15.000        identical
            calibration_bind_r      0.612252      0.612 within tolerance
            calibration_bind_n     12.000000     12.000        identical
                    chi_offset    310.000000    310.000        identical
             megascale_singles 3890

,quantity,this_run,recorded,status
0,ddpca_singles,3688.000000,3688.000,identical
1,ddpca_both_energies,3368.000000,3368.000,identical
2,reliable_binding,2840.000000,2840.000,identical
3,reliable_folding,3109.000000,3109.000,identical
4,reliable_binding_GRB2-SH3,619.000000,619.000,identical
5,reliable_binding_PSD95-PDZ3,1221.000000,1221.000,identical
6,reliable_binding_GB1,1000.000000,1000.000,identical
7,calibration_fold_r,0.829057,0.829,within tolerance
8,calibration_fold_n,15.000000,15.000,identical
9,calibration_bind_r,0.612252,0.612,within tolerance
